# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q panel pandas matplotlib seaborn
    print('Packages installed.')


# Building Interactive Dashboards with Panel

**Panel** is a powerful library that lets you create custom interactive web apps and dashboards by connecting user-defined widgets to plots, images, tables, or text.

Unlike Streamlit (which reruns the whole script), Panel is designed to work well within Jupyter Notebooks and can also be deployed as a standalone app.

In this notebook, we will build a dashboard to explore a synthetic dataset of **Public Opinion Polling**.

In [ ]:
import panel as pn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pn.extension() # Initialize Panel
print("Panel initialized.")

## 1. Create Synthetic Data

We'll simulate polling data across different regions and years.

In [ ]:
years = [2020, 2021, 2022, 2023]
regions = ['North', 'South', 'East', 'West']
topics = ['Economy', 'Health', 'Environment', 'Education']

data_list = []
for year in years:
    for region in regions:
        for topic in topics:
            approval = np.random.randint(30, 80) # Approval rating %
            data_list.append([year, region, topic, approval])

df = pd.DataFrame(data_list, columns=['Year', 'Region', 'Topic', 'Approval'])
df.head()

## 2. Define Widgets

We want the user to filter by **Year** and **Topic**.

In [ ]:
year_slider = pn.widgets.IntSlider(name='Year', start=2020, end=2023, step=1, value=2023)
topic_select = pn.widgets.Select(name='Topic', options=topics, value='Economy')

# Display widgets to check
pn.Row(year_slider, topic_select)

## 3. Define Reactive Plot Function

We create a function that takes the widget values as input and returns a Matplotlib/Seaborn plot. The `@pn.depends` decorator tells Panel to re-run this function whenever the widgets change.

In [ ]:
@pn.depends(year=year_slider, topic=topic_select)
def get_plot(year, topic):
    # Filter data
    subset = df[(df['Year'] == year) & (df['Topic'] == topic)]
    
    # Create Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(x='Region', y='Approval', data=subset, palette='viridis', ax=ax)
    ax.set_ylim(0, 100)
    ax.set_title(f"Approval Ratings for {topic} in {year}")
    ax.set_ylabel("Approval (%)")
    plt.tight_layout()
    plt.close(fig) # Prevent double display in notebook
    return fig

## 4. Assemble the Dashboard

We can arrange the widgets and the plot in a layout (Rows and Columns).

In [ ]:
dashboard = pn.Column(
    "# Public Opinion Dashboard",
    "Use the controls below to filter the data.",
    pn.Row(year_slider, topic_select),
    get_plot
)

# In a notebook, this renders the live dashboard
dashboard.servable()

## 5. Deployment Note

To run this as a standalone web app, you would save this notebook and run the following command in your terminal:

`panel serve 67_dashboard_with_panel.ipynb`

This would launch a local server (usually on port 5006) where you can view the app in your browser without the code cells.